In [22]:
import os
from dotenv import load_dotenv
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
import json
# from recipe_filter import filter_allergens_in_variants, filter_and_sort_recipes
import ast

# Load environment variables
load_dotenv(override=True)

gemini_api_key = os.getenv('GOOGLE_API_KEY')

# Ensure your Google API key is set
genai.configure(api_key=gemini_api_key)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-fin"

# Load the embedding model (same as used for storing data)
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect Pinecone to LangChain
vectorstore = LangChainPinecone(pc.Index(index_name), embed_model, text_key="text")

# Initialize ChatGoogleGenerativeAI for gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Function to generate responses
def generate_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    # Generate content based on the prompt
    response = model.generate_content(prompt)
    return response.text

# Function to filter and sort recipes
def filter_recipes(vectorstore, user_allergens, user_dislikes, query, meal_category, size, protein_option, protein_category, top_k):
    pinecone_filter = {
        "meal_category": {"$eq": meal_category},  # Filter for specific meal type
        # "size": {"$eq": size},  # Filter for specific size
        # "protein_option": {"$eq": protein_option},  # Filter for specific protein option
        # "protein_category": {"$eq": protein_category},  # Filter for specific protein category
        "allergens": {"$nin": list(user_allergens)},  # Exclude recipes containing allergens
        "ingredients": {"$nin": list(user_allergens)}  # Exclude recipes containing allergens
    }
     # ✅ Add `size` filter only if it's not empty
    if size:
        pinecone_filter["size"] = {"$eq": size}

    # ✅ Add `protein_option` filter only if it's not empty
    if protein_category:
        pinecone_filter["protein_category"] = {"$eq": protein_category}

    # ✅ Add `protein_option` filter only if it's not empty
    if protein_option:
        pinecone_filter["protein_option"] = {"$eq": protein_option}
    # Retrieve documents from Pinecone with filtering for dislikes and allergens in ingredients
    docs = vectorstore.similarity_search(
        query=query,
        k=top_k,  # Fetch only the required number of results
        filter=pinecone_filter  # Apply the filter for dislikes and allergens in ingredients
    )


    return docs


# Function to format the filtered recipes into a structured meal plan prompt
def format_meal_plan_prompt(filtered_docs, query, user_likes, user_pref):
    prompt = f"""Generate a weekly meal plan based on the following user persona and recipe data:
    Only use the recipes exactly as they are presented below. Do not modify the recipes, add any extra information, or create new recipes. 
    Simply output the meal name for each day, using only the recipes provided. Do not alter or adapt the meals.
    May include those recipes that have disliked ingredients only if the recipes are not sufficient.
    If the available recipes are insufficient, you may include recipes that contain disliked ingredients. However, you must strictly follow these rules when checking for disliked ingredients:

    1. **ONLY check for disliked ingredients listed in the user's dislikes below. Do NOT flag any ingredient that is NOT in the dislikes list.**
    2. **You must NOT infer, assume, or guess the presence of any ingredient. Only check the explicit list of ingredients provided in the recipe.**
    3. **If a disliked ingredient (from the list below) is found in a meal, append the following notation to the meal: ' * (contains <disliked ingredient>)'.**
    4. **If a meal does not contain any disliked ingredients from the list below, do NOT append anything.**
    5. **Do NOT flag Butter, Cheddar Cheese, Yoghurt, Mascarpone Cheese, Mayonnaise, or any other ingredient unless they are explicitly listed in the dislikes section below and the ingredients list of the recipe.**
    6. **Ensure that the output is in JSON format only.**

    **Meals must be assigned to their respective categories: Breakfast for Breakfast, and Lunch and Dinner should be selected only from the Meal category, with Snacks for Snacks.**
    
    Give response in json format only.
        User Persona:
         Dietary Restrictions: {user_allergens}
         Dislikes: {user_dislikes}
         Likes: {user_likes} 
         Spice Level: Medium 
         Popular Dishes: {user_pref}
         Meal Frequency: 5 meals per day
         Meal Categories: Breakfast, Lunch, Dinner, 2 Snacks  
        """
    for i, doc in enumerate(filtered_docs, 1):
        metadata = doc.metadata

        prompt += f"Meal {i}:\n"
        prompt += f" Dish Name: {metadata.get('dish_name', 'Unknown')}\n"
        prompt += f" Description: {metadata.get('description', 'No description')}\n"
        prompt += f" Ingredients: {', '.join(metadata.get('ingredients', []))}\n"
        prompt += f" Spice Level: {metadata.get('spice_level', 'Not specified')}\n"
        prompt += f" Cuisine: {metadata.get('cuisine', 'Unknown')}\n"
        prompt += f" Meal Category: {metadata.get('meal_category', 'Unknown')}\n"
        # prompt += f" Variants: {metadata.get('variants', 'Unknown')}\n"
    
    return prompt

# Main function to generate the meal plan
def generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref,size, protein_option, protein_category ):
   
    # Step 1: Fetch exact numbers of recipes for each category directly from Pinecone
    breakfast_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query,"breakfast", size, protein_option, protein_category,  8)
    meal_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "meal", size, protein_option, protein_category, 16)
    snack_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query,"snack", size, protein_option, protein_category,  16)

    # Step 2: Organize the filtered recipes by meal category
    # breakfast_docs, meal_docs, snack_docs = organize_recipes_by_category(docs_sorted)
      # Debugging: Print fetched recipe counts
    print(f"Breakfast recipes: {len(breakfast_docs)}")
    print(f"Meal recipes: {len(meal_docs)}")
    print(f"Snack recipes: {len(snack_docs)}")
    print("Fetching complete.")

# Check if the total number of recipes is less than expected
    total_recipes = len(breakfast_docs) + len(meal_docs) + len(snack_docs)
    
    # If the total number of recipes is too few, reset user_allergens and filter again without it
    if total_recipes < (4 + 10 + 10):  # If the number of recipes is less than expected (8 + 16 + 16)
        print("Not enough recipes found. Retrying without allergens filter.")
        user_allergens = {}  # Reset allergens
        # Re-fetch recipes without allergen filter
        breakfast_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query,"breakfast", size, protein_option, protein_category,  8)
        meal_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "meal", size, protein_option, protein_category, 16)
        snack_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query,"snack", size, protein_option, protein_category,  16)

    # Step 2: Combine all selected recipes into the final list
    final_docs = breakfast_docs + meal_docs + snack_docs

    # Step 3: Format the structured meal plan prompt
    final_prompt = format_meal_plan_prompt(final_docs, query, user_likes, user_pref)

    # # Step 4: Generate the response using LLM
    # meal_plan = generate_response(final_prompt)

    # print(meal_plan)
    # return meal_plan, final_docs
    print(final_prompt)

# Example usage
if __name__ == "__main__":
    # User preferences (replace with dynamic input if needed)
    user_allergens = {}  # Example allergens
    # user_dislikes = {
    #     "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
    #     "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
    #     "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
    #     "White Fish", "Worcestershire Sauce"
    # }  # Example disliked ingredients
    
    # query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

    # Generate the meal plan
    # generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)

In [ ]:
user_allergens = {}  # Example allergens
user_pref = "Raspberry Yoghurt, Chocolate Muffin, Cheesy Omelette with Broccoli, Cajun Protein with Spinach, Chicken a la King & Onion Bread Roll"
user_likes = "Mediterranean,European cuisine,Comfort food"
user_dislikes = {
    "Chicken",
    "Chicken Stock Powder",
    "Chicken Jus",
    "Chicken Sausage"
}  # Example disliked ingredients
size = ""
protein_option = ""
protein_category = "low"

# size = ""
# protein_option = ""
# protein_category = ""
query = "Spice Level: Low, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Raspberry Yoghurt, Chocolate Muffin, Cheesy Omelette with Broccoli, Cajun Protein with Spinach, Chicken a la King & Onion Bread Roll"

meal_plan_data= generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref,size, protein_option, protein_category )
# final_docs

Breakfast recipes: 8
Meal recipes: 0
Snack recipes: 16
Fetching complete.
Generate a weekly meal plan based on the following user persona and recipe data:
    Only use the recipes exactly as they are presented below. Do not modify the recipes, add any extra information, or create new recipes. 
    Simply output the meal name for each day, using only the recipes provided. Do not alter or adapt the meals.
    May include those recipes that have disliked ingredients only if the recipes are not sufficient.
    If the available recipes are insufficient, you may include recipes that contain disliked ingredients. However, you must strictly follow these rules when checking for disliked ingredients:

    1. **ONLY check for disliked ingredients listed in the user's dislikes below. Do NOT flag any ingredient that is NOT in the dislikes list.**
    2. **You must NOT infer, assume, or guess the presence of any ingredient. Only check the explicit list of ingredients provided in the recipe.**
    3.